---
## TUGAS MANDIRI (Dikerjakan Selama 1 Minggu)

> **Tenggat waktu:** dikumpulkan paling lambat **sebelum UTS dimulai** (Pertemuan 8 adalah UTS, tidak ada praktikum — namun tugas ini tetap wajib dikumpulkan sesuai tenggat agar tidak menumpuk menjelang pertemuan berikutnya).
> **Sifat tugas:** individu.

### Konteks / Skenario

Kalian direkrut sebagai data analyst paruh waktu oleh **"Kopi Nusantara"**, jaringan kedai kopi dengan beberapa cabang. Pemilik ingin memahami performa penjualan tiap cabang, menu terlaris, dan kontribusi barista kepala di masing-masing cabang selama periode tertentu — persis seperti studi kasus yang baru dikerjakan bersama, namun dengan data dan struktur yang sedikit berbeda.

### Menyiapkan Dataset

Jalankan cell berikut untuk menghasilkan tiga sumber data dan mengunggah data transaksi ke HDFS.

In [1]:
import numpy as np
import pandas as pd
import json

np.random.seed(123)

# Sumber 1: Data cabang (JSON)
cabang_list = ["Kopi Nusantara Magelang", "Kopi Nusantara Yogyakarta", "Kopi Nusantara Semarang"]
data_cabang = [
    {"cabang_id": i + 1, "nama_cabang": c, "kota": c.split()[-1],
     "kepala_barista": np.random.choice(["Wulan", "Reza", "Nadia"])}
    for i, c in enumerate(cabang_list)
]
with open("tugas7_cabang.json", "w") as f:
    for c in data_cabang:
        f.write(json.dumps(c) + "\n")

# Sumber 2: Data menu (CSV)
menu_list = [
    {"menu_id": i, "nama_menu": f"Menu-{i}",
     "kategori_menu": np.random.choice(["Kopi", "Non-Kopi", "Makanan Ringan"]),
     "harga": int(np.random.choice([18000, 22000, 25000, 30000, 35000]))}
    for i in range(1, 16)
]
pd.DataFrame(menu_list).to_csv("tugas7_menu.csv", index=False)

# Sumber 3: Data transaksi (CSV) — data utama
n = 4000
data_transaksi_tugas7 = pd.DataFrame({
    "trx_id": [f"KN{i}" for i in range(n)],
    "cabang_id": np.random.randint(1, 4, size=n),
    "menu_id": np.random.randint(1, 16, size=n),
    "qty": np.random.randint(1, 5, size=n),
})
data_transaksi_tugas7.to_csv("tugas7_transaksi.csv", index=False)

!hdfs dfs -mkdir -p /user/mahasiswa/tugas7/raw
!hdfs dfs -put -f tugas7_transaksi.csv /user/mahasiswa/tugas7/raw/
!hdfs dfs -put -f tugas7_cabang.json /user/mahasiswa/tugas7/raw/
!hdfs dfs -put -f tugas7_menu.csv /user/mahasiswa/tugas7/raw/

print("Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.")
print(f"- Cabang: {len(data_cabang)} | Menu: {len(menu_list)} | Transaksi: {n}")

Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.
- Cabang: 3 | Menu: 15 | Transaksi: 4000


### Instruksi Pengerjaan

Buat notebook baru **`Tugas7_[NPM]_[Nama Lengkap].ipynb`**, lalu bangun pipeline analisis lengkap:

---

**A. EXTRACT** *(bobot 10%)*

Baca ketiga sumber data **dari HDFS** (bukan disk lokal) menjadi tiga Spark DataFrame.

**B. TRANSFORM — Join & Kolom Turunan** *(bobot 20%)*

Gabungkan ketiga tabel, tambahkan kolom `total_penjualan` (`qty x harga`).

**C. TRANSFORM — Window Function** *(bobot 20%)*

Tentukan **top-2 menu terlaris** (berdasarkan `total_penjualan`) di **setiap cabang**, gunakan `row_number()`.

**D. TRANSFORM — Spark SQL** *(bobot 20%)*

Menggunakan **Spark SQL murni**, tampilkan total penjualan per `kepala_barista`, diurutkan dari tertinggi.

**E. LOAD** *(bobot 15%)*

Simpan data gabungan (hasil bagian B) ke HDFS sebagai Parquet, path `/user/[username]/tugas7/processed/laporan`, dipartisi berdasarkan `kategori_menu`. Verifikasi dengan membaca kembali dan menghitung jumlah baris.

**F. Ringkasan Eksekutif & Rekomendasi** *(bobot 15%)*

Buat tabel ringkasan (per cabang: total penjualan, jumlah transaksi, rata-rata nilai transaksi), lalu tulis pada markdown cell (**minimal 120 kata**): cabang mana yang berkinerja terbaik, menu apa yang sebaiknya dipromosikan lebih gencar di cabang dengan kinerja terlemah (berdasarkan temuan bagian C), dan alasannya.

---

### Ketentuan Pengumpulan

- Kumpulkan `Tugas7_[NPM]_[Nama Lengkap].ipynb` melalui Asprak, paling lambat **sebelum Pertemuan UTS dimulai**.
- Pastikan Hadoop aktif dan seluruh cell sudah dijalankan (**Run All**) sebelum dikumpulkan.

### Rubrik Penilaian

| Bagian | Kriteria | Bobot |
|---|---|---|
| A. Extract | Ketiga sumber berhasil dibaca dari HDFS | 10% |
| B. Join & Transformasi | Join benar, kolom `total_penjualan` tepat | 20% |
| C. Window Function | Top-2 menu per cabang benar menggunakan `row_number()` | 20% |
| D. Spark SQL | Kueri SQL murni berjalan benar sesuai instruksi | 20% |
| E. Load | Data tersimpan ke HDFS sebagai Parquet terpartisi & terverifikasi | 15% |
| F. Ringkasan & Rekomendasi | Insight berbasis data, rekomendasi logis dan relevan | 15% |


### Mengaktifkan SparkSession

In [2]:
#Import fungsi yang dipakai di seluruh pipeline
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, row_number, round as spark_round
from pyspark.sql.window import Window

#SparkSession = pintu masuk ke semua fitur Spark
spark = SparkSession.builder \
    .appName("Tugas7-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()                        #local[*] = pakai semua core CPU
spark.sparkContext.setLogLevel("ERROR")   #sembunyikan log selain error

print("SparkSession siap. Versi Spark:", spark.version)

26/09/29 19:34:03 WARN Utils: Your hostname, ThinkpadT14-VirtualBox resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/09/29 19:34:03 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/29 19:34:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession siap. Versi Spark: 3.5.9


## A. Tahap EXTRACT 

In [3]:
#Baca 3 sumber data DARI HDFS (bukan disk lokal)
#header=True: baris pertama jadi nama kolom; inferSchema=True: tipe data ditebak otomatis
df_transaksi = spark.read.csv(
    "hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_transaksi.csv",
    header=True, inferSchema=True
)
df_cabang = spark.read.json("hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_cabang.json")
df_menu = spark.read.csv(
    "hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_menu.csv",
    header=True, inferSchema=True
)

#Cek jumlah baris (harus 4000 | 3 | 15)
print("Transaksi:", df_transaksi.count(), "| Cabang:", df_cabang.count(), "| Menu:", df_menu.count())

Transaksi: 4000 | Cabang: 3 | Menu: 15


## B. TRANSFORM — Join & Kolom Turunan 

In [4]:
#Gabungkan transaksi dengan cabang (kunci: cabang_id), lalu dengan menu (kunci: menu_id)
#inner join: hanya baris yang punya pasangan di kedua tabel yang dipertahankan
df_gabungan = df_transaksi.join(df_cabang, on="cabang_id", how="inner") \
                          .join(df_menu, on="menu_id", how="inner")

#Kolom baru: total_penjualan = jumlah unit x harga satuan
df_gabungan = df_gabungan.withColumn("total_penjualan", col("qty") * col("harga"))
df_gabungan.show(5)

+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|menu_id|cabang_id|trx_id|qty|kepala_barista|      kota|         nama_cabang|nama_menu| kategori_menu|harga|total_penjualan|
+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|     14|        1|   KN0|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-14|          Kopi|18000|          72000|
|     11|        1|   KN1|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          88000|
|     11|        1|   KN2|  3|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          66000|
|     11|        3|   KN3|  1|         Nadia|  Semarang|Kopi Nusantara Se...|  Menu-11|          Kopi|22000|          22000|
|     12|        2|   KN4|  4|          Reza|Yogyakarta|Kopi Nusantara Yo...|  Menu-12|Makanan Ringan|35000|         140000|


## C. TRANSFORM — Window Function

In [5]:
#Langkah 1: ringkas jadi satu baris per kombinasi (cabang, menu)
ringkasan_menu_cabang = df_gabungan.groupBy("nama_cabang", "nama_menu", "kategori_menu").agg(
    spark_sum("total_penjualan").alias("total_penjualan")
)

#Langkah 2: window dibagi per cabang, diurutkan dari penjualan terbesar
window_cabang = Window.partitionBy("nama_cabang").orderBy(col("total_penjualan").desc())

#Langkah 3: beri nomor urut di tiap cabang dengan row_number(), simpan nomor 1 dan 2
top2_menu = ringkasan_menu_cabang.withColumn("peringkat", row_number().over(window_cabang)) \
                                 .filter(col("peringkat") <= 2)

top2_menu.orderBy("nama_cabang", "peringkat").show(truncate=False)

[Stage 21:>                                                         (0 + 1) / 1]

+-------------------------+---------+--------------+---------------+---------+
|nama_cabang              |nama_menu|kategori_menu |total_penjualan|peringkat|
+-------------------------+---------+--------------+---------------+---------+
|Kopi Nusantara Magelang  |Menu-1   |Makanan Ringan|8505000        |1        |
|Kopi Nusantara Magelang  |Menu-12  |Makanan Ringan|8085000        |2        |
|Kopi Nusantara Semarang  |Menu-13  |Makanan Ringan|8750000        |1        |
|Kopi Nusantara Semarang  |Menu-9   |Makanan Ringan|8715000        |2        |
|Kopi Nusantara Yogyakarta|Menu-12  |Makanan Ringan|9275000        |1        |
|Kopi Nusantara Yogyakarta|Menu-15  |Non-Kopi      |8160000        |2        |
+-------------------------+---------+--------------+---------------+---------+



## D. TRANSFORM — Spark SQL 

In [6]:
#Daftarkan DataFrame sebagai tabel sementara bernama gabungan_kopi
df_gabungan.createOrReplaceTempView("gabungan_kopi")

penjualan_barista = spark.sql('''
    SELECT kepala_barista, SUM(total_penjualan) AS total_penjualan   -- total rupiah per barista
    FROM gabungan_kopi
    GROUP BY kepala_barista
    ORDER BY total_penjualan DESC                                    -- tertinggi di atas
''')
penjualan_barista.show()

+--------------+---------------+
|kepala_barista|total_penjualan|
+--------------+---------------+
|         Nadia|      177680000|
|          Reza|       84645000|
+--------------+---------------+



## E. LOAD 

In [7]:
#Buat folder tujuan di HDFS
!hdfs dfs -mkdir -p /user/mahasiswa/tugas7/processed

#partitionBy: Spark membuat 1 subfolder per kategori_menu (Kopi, Non-Kopi, Makanan Ringan)
#overwrite: jika folder sudah ada, ditimpa (aman dijalankan berulang)
df_gabungan.write.mode("overwrite").partitionBy("kategori_menu").parquet(
    "hdfs://localhost:9000/user/mahasiswa/tugas7/processed/laporan"
)

#Lihat struktur folder hasilnya
!hdfs dfs -ls /user/mahasiswa/tugas7/processed/laporan

#Verifikasi: baca kembali dari HDFS dan hitung jumlah barisnya (harus 4000)
df_verifikasi = spark.read.parquet("hdfs://localhost:9000/user/mahasiswa/tugas7/processed/laporan")
print("Jumlah baris dibaca ulang:", df_verifikasi.count())

Found 4 items
-rw-r--r--   3 dian-khaerunia-azzahra supergroup          0 2026-09-29 19:35 /user/mahasiswa/tugas7/processed/laporan/_SUCCESS
drwxr-xr-x   - dian-khaerunia-azzahra supergroup          0 2026-09-29 19:35 /user/mahasiswa/tugas7/processed/laporan/kategori_menu=Kopi
drwxr-xr-x   - dian-khaerunia-azzahra supergroup          0 2026-09-29 19:35 /user/mahasiswa/tugas7/processed/laporan/kategori_menu=Makanan Ringan
drwxr-xr-x   - dian-khaerunia-azzahra supergroup          0 2026-09-29 19:35 /user/mahasiswa/tugas7/processed/laporan/kategori_menu=Non-Kopi


[Stage 34:>                                                         (0 + 3) / 3]

Jumlah baris dibaca ulang: 4000


## F. Ringkasan Eksekutif & Rekomendasi 

In [8]:
#Ringkasan per cabang, dibaca dari hasil Parquet di HDFS
ringkasan_eksekutif = df_verifikasi.groupBy("nama_cabang", "kota", "kepala_barista").agg(
    spark_sum("total_penjualan").alias("total_penjualan"),                      #total rupiah
    count("trx_id").alias("jumlah_transaksi"),                                  #jumlah transaksi
    spark_round(avg("total_penjualan"), 0).alias("rata_rata_nilai_transaksi")   #rata-rata, dibulatkan
).orderBy(col("total_penjualan").desc())                                        #cabang terbaik di atas

ringkasan_eksekutif.show(truncate=False)

+-------------------------+----------+--------------+---------------+----------------+-------------------------+
|nama_cabang              |kota      |kepala_barista|total_penjualan|jumlah_transaksi|rata_rata_nilai_transaksi|
+-------------------------+----------+--------------+---------------+----------------+-------------------------+
|Kopi Nusantara Semarang  |Semarang  |Nadia         |90542000       |1393            |64998.0                  |
|Kopi Nusantara Magelang  |Magelang  |Nadia         |87138000       |1302            |66926.0                  |
|Kopi Nusantara Yogyakarta|Yogyakarta|Reza          |84645000       |1305            |64862.0                  |
+-------------------------+----------+--------------+---------------+----------------+-------------------------+



Berdasarkan tabel ringkasan, cabang dengan kinerja terbaik adalah Kopi Nusantara Semarang dengan total penjualan Rp90.542.000 dari 1.393 transaksi, disusul Magelang sebesar Rp87.138.000 dan Yogyakarta sebesar Rp84.645.000. Keunggulan Semarang terutama datang dari jumlah transaksinya yang paling banyak, karena rata-rata nilai per transaksinya (sekitar Rp64.998) justru lebih rendah daripada Magelang (sekitar Rp66.926). Cabang dengan kinerja terlemah adalah Yogyakarta, yang totalnya sekitar Rp5,9 juta di bawah Semarang dan rata-rata nilai transaksinya juga paling kecil, yaitu sekitar Rp64.862.

Dari hasil bagian C, dua menu terlaris di Yogyakarta adalah Menu-12 (Makanan Ringan) dengan penjualan Rp9.275.000 dan Menu-15 (Non-Kopi) dengan Rp8.160.000. Menurut saya kedua menu ini yang sebaiknya dipromosikan lebih gencar di Yogyakarta, misalnya lewat paket bundling atau diskon pembelian kedua, karena keduanya sudah terbukti paling diminati pelanggan di sana sehingga peluang berhasilnya lebih besar dibanding mempromosikan menu yang belum populer. Menu-12 juga masuk tiga besar di Magelang dan Semarang, jadi menu ini terlihat cukup disukai di semua cabang. Promosi bundling juga bisa membantu menaikkan nilai per transaksi, yang memang menjadi kelemahan Yogyakarta. Meski begitu, selisih antar cabang cukup tipis, jadi hasilnya sebaiknya dipantau beberapa periode dulu sebelum dijadikan keputusan besar.

### Menutup SparkSession

In [9]:
#Tutup SparkSession setelah semua selesai
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
